# 27e — Partial Master Salvage and Target Readiness

## Objetivo

Aprovechar de forma correcta las filas negativas cuya X sí fue recuperada en `27d`, sin bloquear toda la reconstrucción del master por dos repositorios incompletos.

El notebook `27d` ya validó el extractor contra los dos sitios congelados, pero obtuvo X completa en **10 de 12** repositorios negativos.

La regla correcta para el dataset maestro es:

```text
fila con X completa
→ puede incorporarse

fila con X incompleta
→ se excluye de ese espacio X
→ no se convierte a cero
→ queda registrada para recuperación posterior
```

Esto es consistente con el diseño semisupervisado original: una fila no válida no debe impedir utilizar otras filas válidas.

Este notebook:

1. verifica que el extractor de `27d` fue validado;
2. separa filas recuperadas y filas incompletas;
3. une las filas completas con `catching_targets.csv`;
4. conserva `NaN` de Y como desconocido, target por target;
5. reconstruye el master v2;
6. recalcula L/U;
7. audita soporte 0/1 por heurística;
8. identifica exactamente qué filas incompletas bloquean H1/H4/H5/H6;
9. decide si el notebook 28 puede ser completo, parcial o debe seguir precedido por recuperación técnica.

No entrena modelos.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_DIR = PROJECT_DIR / "data" / "semisupervised"
SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
SEMISUP_INTERIM_DIR = SEMISUP_DIR / "interim"

SEMISUP_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SEMISUP_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

FEATURES = [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar artefactos de 27d

In [2]:
negative_features_path = (
    SEMISUP_INTERIM_DIR
    / "catching_negative_static_site_features_v2.csv"
)

queue_path = (
    SEMISUP_REPORTS_DIR
    / "catching_negative_feature_extraction_queue.csv"
)

targets_path = (
    PUBLIC_INTERIM_DIR
    / "catching_targets.csv"
)

master_v1_path = (
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset.csv"
)

regression_path = (
    SEMISUP_REPORTS_DIR
    / "catching_extractor_regression_validation_v2.csv"
)

decision_27d_path = (
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_decision.csv"
)

required_paths = [
    negative_features_path,
    queue_path,
    targets_path,
    master_v1_path,
    regression_path
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

assert missing == [], missing

negative_features = pd.read_csv(
    negative_features_path
)

queue = pd.read_csv(
    queue_path
)

targets = pd.read_csv(
    targets_path
)

master_v1 = pd.read_csv(
    master_v1_path
)

regression = pd.read_csv(
    regression_path
)

decision_27d = (
    pd.read_csv(decision_27d_path)
    if decision_27d_path.exists()
    else pd.DataFrame()
)

print("Negative features:", negative_features.shape)
print("Queue:", queue.shape)
print("Targets:", targets.shape)
print("Master v1:", master_v1.shape)
print("Regression:", regression.shape)


Negative features: (12, 12)
Queue: (12, 10)
Targets: (30, 8)
Master v1: (1002, 22)
Regression: (2, 11)


## 3. Verificar que el extractor sí quedó validado

In [3]:
required_regression_cols = {
    "depth_match",
    "interactive_match",
    "label_ratio_match",
    "all_three_match"
}

assert required_regression_cols.issubset(
    regression.columns
)

extractor_validated = bool(
    len(regression) > 0
    and
    regression[
        "all_three_match"
    ].fillna(False).astype(bool).all()
)

print(
    "EXTRACTOR_VALIDATED:",
    extractor_validated
)

display(
    regression
)

assert extractor_validated is True


EXTRACTOR_VALIDATED: True


,site,depth_frozen,depth_new,interactive_frozen,interactive_new,label_ratio_frozen,label_ratio_new,depth_match,interactive_match,label_ratio_match,all_three_match
0,startbootstrap-3-col-portfolio,8.0,8,23.0,23,0.739130,0.739130,True,True,True,True
1,startbootstrap-sb-admin-2,15.0,15,585.0,585,0.842735,0.842735,True,True,True,True


## 4. Clasificar filas por completitud de X

Una fila se considera utilizable solo cuando las tres variables proxy están disponibles:

```text
static_max_dom_depth
static_total_interactive_controls
static_control_label_proxy_ratio
```

No se imputa ningún `NaN`.


In [4]:
source_feature_columns = [
    "static_max_dom_depth",
    "static_total_interactive_controls",
    "static_control_label_proxy_ratio"
]

missing_feature_columns = sorted(
    set(source_feature_columns)
    -
    set(negative_features.columns)
)

assert missing_feature_columns == [], missing_feature_columns

for col in source_feature_columns:
    negative_features[col] = pd.to_numeric(
        negative_features[col],
        errors="coerce"
    )

negative_features[
    "complete_proxy_x"
] = negative_features[
    source_feature_columns
].notna().all(axis=1)

usable_negative_features = (
    negative_features[
        negative_features[
            "complete_proxy_x"
        ]
    ]
    .copy()
)

excluded_negative_features = (
    negative_features[
        ~negative_features[
            "complete_proxy_x"
        ]
    ]
    .copy()
)

print(
    "Filas recuperadas totales:",
    len(negative_features)
)

print(
    "Filas con X completa:",
    len(usable_negative_features)
)

print(
    "Filas excluidas por X incompleta:",
    len(excluded_negative_features)
)

display(
    excluded_negative_features
)


Filas recuperadas totales: 12
Filas con X completa: 10
Filas excluidas por X incompleta: 2


,site,repo_full_name,static_page_count,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,extract_ok,extract_reason,source_match_level,strict_source_match,extended_source_match,complete_proxy_x
3,react-saas-template,dunky11/react-saas-template,1,2.0,0.0,NaN,True,NaN,B_MODERATE_VALIDATED,False,True,False
4,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,0,NaN,NaN,NaN,False,NO_HTML_FILES,C_EXACT_NAME_TECHNICAL,False,True,False


## 5. Adjuntar metadata de la cola de recuperación

In [5]:
queue_meta_cols = [
    "site_or_instance",
    "negative_targets",
    "repository",
    "evidence_tier",
    "strict_source_match",
    "extended_source_match"
]

queue_meta_cols = [
    col
    for col in queue_meta_cols
    if col in queue.columns
]

queue_meta = (
    queue[
        queue_meta_cols
    ]
    .drop_duplicates(
        subset=[
            "site_or_instance"
        ]
    )
    .rename(
        columns={
            "site_or_instance":
                "site"
        }
    )
)

usable_negative_features = (
    usable_negative_features
    .merge(
        queue_meta,
        on="site",
        how="left",
        suffixes=(
            "",
            "_queue"
        ),
        validate="one_to_one"
    )
)

excluded_negative_features = (
    excluded_negative_features
    .merge(
        queue_meta,
        on="site",
        how="left",
        suffixes=(
            "",
            "_queue"
        ),
        validate="one_to_one"
    )
)

display(
    usable_negative_features[
        [
            "site",
            "negative_targets",
            "source_match_level",
            "complete_proxy_x"
        ]
    ]
)

display(
    excluded_negative_features[
        [
            "site",
            "negative_targets",
            "extract_reason",
            "static_max_dom_depth",
            "static_total_interactive_controls",
            "static_control_label_proxy_ratio"
        ]
    ]
)


,site,negative_targets,source_match_level,complete_proxy_x
0,beginner-html-site-styled,H6_violation,B_MODERATE_VALIDATED,True
1,conference-website-template,H6_violation,B_MODERATE_VALIDATED,True
2,personal-website-template,H6_violation,B_MODERATE_VALIDATED,True
3,charity_responsive-template,H6_violation,C_EXACT_NAME_TECHNICAL,True
4,simple-gallery_responsive-template,H6_violation,C_EXACT_NAME_TECHNICAL,True
5,single-page-small-business-web-application,H6_violation,C_EXACT_NAME_TECHNICAL,True
6,startbootstrap-freelancer,H4_violation | H6_violation,C_EXACT_NAME_TECHNICAL,True
7,startbootstrap-landing-page,H6_violation,C_EXACT_NAME_TECHNICAL,True
8,startbootstrap-one-page-wonder,H6_violation,C_EXACT_NAME_TECHNICAL,True
9,startbootstrap-resume,H5_violation,C_EXACT_NAME_TECHNICAL,True


,site,negative_targets,extract_reason,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio
0,react-saas-template,H6_violation,NaN,2.0,0.0,NaN
1,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation | H4_violation | H5_violation | H6_violation,NO_HTML_FILES,NaN,NaN,NaN


## 6. Normalización de identidad y unión con Y

In [6]:
def normalize_identity(value):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower()
    )


targets = targets.copy()

targets[
    "__site_key"
] = targets[
    "site"
].map(
    normalize_identity
)

usable_negative_features[
    "__site_key"
] = usable_negative_features[
    "site"
].map(
    normalize_identity
)

excluded_negative_features[
    "__site_key"
] = excluded_negative_features[
    "site"
].map(
    normalize_identity
)

target_columns = [
    "site",
    "__site_key",
    "source_dataset",
    "label_source",
    "label_level"
] + TARGETS

target_columns = [
    col
    for col in target_columns
    if col in targets.columns
]

usable_xy = (
    usable_negative_features
    .merge(
        targets[
            target_columns
        ],
        on="__site_key",
        how="left",
        suffixes=(
            "_feature",
            "_target"
        ),
        validate="one_to_one"
    )
)

excluded_xy = (
    excluded_negative_features
    .merge(
        targets[
            target_columns
        ],
        on="__site_key",
        how="left",
        suffixes=(
            "_feature",
            "_target"
        ),
        validate="one_to_one"
    )
)

print(
    "Usable X/Y rows:",
    len(usable_xy)
)

display(
    usable_xy
)

print(
    "Excluded rows with their true Y:",
    len(excluded_xy)
)

display(
    excluded_xy
)


Usable X/Y rows: 10


,site_feature,repo_full_name,static_page_count,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,extract_ok,extract_reason,source_match_level,strict_source_match,extended_source_match,complete_proxy_x,negative_targets,repository,evidence_tier,strict_source_match_queue,extended_source_match_queue,__site_key,site_target,source_dataset,label_source,label_level,H1_violation,H4_violation,H5_violation,H6_violation
0,beginner-html-site-styled,mdn/beginner-html-site-styled,1,3.0,1.0,1.000000,True,NaN,B_MODERATE_VALIDATED,False,True,True,H6_violation,mdn/beginner-html-site-styled,B_MODERATE_VALIDATED,False,True,beginnerhtmlsitestyled,beginner-html-site-styled,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,NaN,1,0
1,conference-website-template,mikepierce/conference-website-template,5,6.0,41.0,0.878049,True,NaN,B_MODERATE_VALIDATED,False,True,True,H6_violation,mikepierce/conference-website-template,B_MODERATE_VALIDATED,False,True,conferencewebsitetemplate,conference-website-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
2,personal-website-template,The-WebOps-Club/personal-website-template,1,11.0,48.0,0.729167,True,NaN,B_MODERATE_VALIDATED,False,True,True,H6_violation,The-WebOps-Club/personal-website-template,B_MODERATE_VALIDATED,False,True,personalwebsitetemplate,personal-website-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
3,charity_responsive-template,website-templates/charity_responsive-template,1,7.0,25.0,0.840000,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H6_violation,website-templates/charity_responsive-template,C_EXACT_NAME_TECHNICAL,False,True,charityresponsivetemplate,charity_responsive-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
4,simple-gallery_responsive-template,website-templates/simple-gallery_responsive-template,3,15.0,38.0,0.973684,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H6_violation,website-templates/simple-gallery_responsive-template,C_EXACT_NAME_TECHNICAL,False,True,simplegalleryresponsivetemplate,simple-gallery_responsive-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
5,single-page-small-business-web-application,reza-azimifar/single-page-small-business-web-application,2,8.0,60.0,0.833333,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H6_violation,reza-azimifar/single-page-small-business-web-application,C_EXACT_NAME_TECHNICAL,False,True,singlepagesmallbusinesswebapplication,single-page-small-business-web-application,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
6,startbootstrap-freelancer,StartBootstrap/startbootstrap-freelancer,1,11.0,29.0,0.862069,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H4_violation | H6_violation,StartBootstrap/startbootstrap-freelancer,C_EXACT_NAME_TECHNICAL,False,True,startbootstrapfreelancer,startbootstrap-freelancer,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,0.0,1,0
7,startbootstrap-landing-page,StartBootstrap/startbootstrap-landing-page,1,10.0,15.0,0.666667,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H6_violation,StartBootstrap/startbootstrap-landing-page,C_EXACT_NAME_TECHNICAL,False,True,startbootstraplandingpage,startbootstrap-landing-page,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
8,startbootstrap-one-page-wonder,StartBootstrap/startbootstrap-one-page-wonder,1,7.0,5.0,1.000000,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H6_violation,StartBootstrap/startbootstrap-one-page-wonder,C_EXACT_NAME_TECHNICAL,False,True,startbootstraponepagewonder,startbootstrap-one-page-wonder,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
9,startbootstrap-resume,StartBootstrap/startbootstrap-resume,1,8.0,13.0,0.692308,True,NaN,C_EXACT_NAME_TECHNICAL,False,True,True,H5_violation,StartBootstrap/startbootstrap-resume,C_EXACT_NAME_TECHNICAL,False,True,startbootstrapresume,startbootstrap-resume,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,0,1


Excluded rows with their true Y: 2


,site_feature,repo_full_name,static_page_count,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,extract_ok,extract_reason,source_match_level,strict_source_match,extended_source_match,complete_proxy_x,negative_targets,repository,evidence_tier,strict_source_match_queue,extended_source_match_queue,__site_key,site_target,source_dataset,label_source,label_level,H1_violation,H4_violation,H5_violation,H6_violation
0,react-saas-template,dunky11/react-saas-template,1,2.0,0.0,NaN,True,NaN,B_MODERATE_VALIDATED,False,True,False,H6_violation,dunky11/react-saas-template,B_MODERATE_VALIDATED,False,True,reactsaastemplate,react-saas-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,0
1,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,0,NaN,NaN,NaN,False,NO_HTML_FILES,C_EXACT_NAME_TECHNICAL,False,True,False,H1_violation | H4_violation | H5_violation | H6_violation,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,C_EXACT_NAME_TECHNICAL,False,True,electronicsecommerceshopwithadmindashboardnextjsnodejsmain,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,0,0.0,0,0


## 7. Auditar Y sin exigir vector completo

Una fila puede ser `L` para una heurística y `U` para otra.

Por eso:

- `0` y `1` son Y observadas válidas;
- `NaN` permanece desconocido;
- cualquier otro valor es inválido.

No se elimina una fila completa porque un solo target tenga `NaN`.


In [7]:
y_audit_records = []

for target in TARGETS:

    values = pd.to_numeric(
        usable_xy[
            target
        ],
        errors="coerce"
    )

    invalid = (
        values.notna()
        &
        ~values.isin(
            [
                0,
                1
            ]
        )
    )

    y_audit_records.append({
        "target":
            target,
        "n_rows":
            len(values),
        "n_observed":
            int(
                values.notna().sum()
            ),
        "n_unknown":
            int(
                values.isna().sum()
            ),
        "n_0":
            int(
                (
                    values
                    ==
                    0
                ).sum()
            ),
        "n_1":
            int(
                (
                    values
                    ==
                    1
                ).sum()
            ),
        "n_invalid":
            int(
                invalid.sum()
            )
    })

usable_y_audit = pd.DataFrame(
    y_audit_records
)

display(
    usable_y_audit
)

assert (
    usable_y_audit[
        "n_invalid"
    ]
    ==
    0
).all()


,target,n_rows,n_observed,n_unknown,n_0,n_1,n_invalid
0,H1_violation,10,10,0,0,10,0
1,H4_violation,10,9,1,1,8,0
2,H5_violation,10,10,0,1,9,0
3,H6_violation,10,10,0,9,1,0


## 8. Construir filas extendidas del master

In [8]:
def normalize_id_part(value):

    value = str(
        value
    ).strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value
    )

    return value.strip("_")


def master_id(
    source_dataset,
    source_instance_id
):

    return (
        normalize_id_part(
            source_dataset
        )
        +
        "__"
        +
        normalize_id_part(
            source_instance_id
        )
    )


extended_rows = []

for _, row in (
    usable_xy.iterrows()
):

    site_value = row[
        "site_feature"
    ]

    extended_rows.append({
        "master_instance_id":
            master_id(
                "Catching_UX_Flaws",
                site_value
            ),

        "source_dataset":
            "Catching_UX_Flaws",

        "source_instance_id":
            str(
                site_value
            ),

        "source_unit":
            "website",

        "feature_domain":
            "WEB_DOM",

        "feature_space":
            "EXPERIMENTAL_CONSTRUCT_PROXY",

        "structure_depth":
            pd.to_numeric(
                row[
                    "static_max_dom_depth"
                ],
                errors="coerce"
            ),

        "interactive_control_count":
            pd.to_numeric(
                row[
                    "static_total_interactive_controls"
                ],
                errors="coerce"
            ),

        "interactive_label_ratio":
            pd.to_numeric(
                row[
                    "static_control_label_proxy_ratio"
                ],
                errors="coerce"
            ),

        "H1_violation":
            pd.to_numeric(
                row[
                    "H1_violation"
                ],
                errors="coerce"
            ),

        "H4_violation":
            pd.to_numeric(
                row[
                    "H4_violation"
                ],
                errors="coerce"
            ),

        "H5_violation":
            pd.to_numeric(
                row[
                    "H5_violation"
                ],
                errors="coerce"
            ),

        "H6_violation":
            pd.to_numeric(
                row[
                    "H6_violation"
                ],
                errors="coerce"
            ),

        "label_source":
            str(
                row.get(
                    "label_source",
                    "Catching"
                )
            ),

        "source_label_level":
            str(
                row.get(
                    "label_level",
                    "LLM_DIRECT"
                )
            ),

        "nielsen_label_level":
            str(
                row.get(
                    "label_level",
                    "LLM_DIRECT"
                )
            ),

        "is_pseudo_label":
            False,

        "group_id":
            (
                "catching__"
                +
                normalize_id_part(
                    site_value
                )
            ),

        "domain_shift_flag":
            "WEB_DOM",

        "complete_proxy_x":
            True,

        "has_any_observed_nielsen_y":
            bool(
                pd.notna(
                    row[
                        TARGETS
                    ]
                ).any()
            ),

        "master_role":
            "OBSERVED_LABEL_SOURCE_EXTENDED",

        "source_match_level":
            row[
                "source_match_level"
            ]
    })

extended_master_rows = pd.DataFrame(
    extended_rows
)

print(
    "Extended master rows:",
    extended_master_rows.shape
)

display(
    extended_master_rows
)


Extended master rows: (10, 23)


,master_instance_id,source_dataset,source_instance_id,source_unit,feature_domain,feature_space,structure_depth,interactive_control_count,interactive_label_ratio,H1_violation,H4_violation,H5_violation,H6_violation,label_source,source_label_level,nielsen_label_level,is_pseudo_label,group_id,domain_shift_flag,complete_proxy_x,has_any_observed_nielsen_y,master_role,source_match_level
0,catching_ux_flaws__beginner_html_site_styled,Catching_UX_Flaws,beginner-html-site-styled,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,3.0,1.0,1.000000,1,NaN,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__beginner_html_site_styled,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,B_MODERATE_VALIDATED
1,catching_ux_flaws__conference_website_template,Catching_UX_Flaws,conference-website-template,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,6.0,41.0,0.878049,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__conference_website_template,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,B_MODERATE_VALIDATED
2,catching_ux_flaws__personal_website_template,Catching_UX_Flaws,personal-website-template,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,11.0,48.0,0.729167,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__personal_website_template,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,B_MODERATE_VALIDATED
3,catching_ux_flaws__charity_responsive_template,Catching_UX_Flaws,charity_responsive-template,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,7.0,25.0,0.840000,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__charity_responsive_template,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL
4,catching_ux_flaws__simple_gallery_responsive_template,Catching_UX_Flaws,simple-gallery_responsive-template,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,15.0,38.0,0.973684,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__simple_gallery_responsive_template,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL
5,catching_ux_flaws__single_page_small_business_web_application,Catching_UX_Flaws,single-page-small-business-web-application,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,8.0,60.0,0.833333,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__single_page_small_business_web_application,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL
6,catching_ux_flaws__startbootstrap_freelancer,Catching_UX_Flaws,startbootstrap-freelancer,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,11.0,29.0,0.862069,1,0.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_freelancer,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL
7,catching_ux_flaws__startbootstrap_landing_page,Catching_UX_Flaws,startbootstrap-landing-page,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,10.0,15.0,0.666667,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_landing_page,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL
8,catching_ux_flaws__startbootstrap_one_page_wonder,Catching_UX_Flaws,startbootstrap-one-page-wonder,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,7.0,5.0,1.000000,1,1.0,1,0,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_one_page_wonder,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL
9,catching_ux_flaws__startbootstrap_resume,Catching_UX_Flaws,startbootstrap-resume,website,WEB_DOM,EXPERIMENTAL_CONSTRUCT_PROXY,8.0,13.0,0.692308,1,1.0,0,1,LLM_CONSENSUS,LLM_DIRECT,LLM_DIRECT,False,catching__startbootstrap_resume,WEB_DOM,True,True,OBSERVED_LABEL_SOURCE_EXTENDED,C_EXACT_NAME_TECHNICAL


## 9. Reconstruir master v2

In [9]:
master_v2_base = master_v1.copy()

if (
    "source_match_level"
    not in
    master_v2_base.columns
):

    master_v2_base[
        "source_match_level"
    ] = np.where(
        master_v2_base[
            "source_dataset"
        ]
        ==
        "Catching_UX_Flaws",
        "A_CONFIRMED_STRONG",
        "NOT_APPLICABLE_UNLABELED"
    )

collision_ids = set(
    master_v2_base[
        "master_instance_id"
    ]
).intersection(
    set(
        extended_master_rows[
            "master_instance_id"
        ]
    )
)

collisions = (
    extended_master_rows[
        extended_master_rows[
            "master_instance_id"
        ].isin(
            collision_ids
        )
    ]
    .copy()
)

safe_extended = (
    extended_master_rows[
        ~extended_master_rows[
            "master_instance_id"
        ].isin(
            collision_ids
        )
    ]
    .copy()
)

master_v2 = pd.concat(
    [
        master_v2_base,
        safe_extended
    ],
    ignore_index=True,
    sort=False
)

master_v2_built = bool(
    extractor_validated
    and
    len(
        safe_extended
    )
    >
    0
    and
    master_v2[
        "master_instance_id"
    ].duplicated().sum()
    ==
    0
)

print(
    "Master v2 built:",
    master_v2_built
)

print(
    "Rows v1:",
    len(
        master_v1
    )
)

print(
    "Rows salvaged:",
    len(
        safe_extended
    )
)

print(
    "Rows v2:",
    len(
        master_v2
    )
)

print(
    "Collisions:",
    len(
        collisions
    )
)

assert master_v2_built is True


Master v2 built: True
Rows v1: 1002
Rows salvaged: 10
Rows v2: 1012
Collisions: 0


## 10. Balance de clases observado en Catching v2

In [10]:
labeled_catching = (
    master_v2[
        master_v2[
            "source_dataset"
        ]
        ==
        "Catching_UX_Flaws"
    ]
    .copy()
)

balance_records = []

for target in TARGETS:

    values = pd.to_numeric(
        labeled_catching[
            target
        ],
        errors="coerce"
    )

    balance_records.append({
        "target":
            target,

        "n_rows_catching":
            len(
                labeled_catching
            ),

        "n_labeled":
            int(
                values.notna().sum()
            ),

        "n_unknown":
            int(
                values.isna().sum()
            ),

        "n_0":
            int(
                (
                    values
                    ==
                    0
                ).sum()
            ),

        "n_1":
            int(
                (
                    values
                    ==
                    1
                ).sum()
            ),

        "n_classes":
            int(
                values.dropna().nunique()
            ),

        "both_classes_observed":
            (
                values.dropna().nunique()
                >=
                2
            )
    })

master_v2_balance = pd.DataFrame(
    balance_records
)

display(
    master_v2_balance
)


,target,n_rows_catching,n_labeled,n_unknown,n_0,n_1,n_classes,both_classes_observed
0,H1_violation,12,12,0,0,12,1,False
1,H4_violation,12,11,1,1,10,2,True
2,H5_violation,12,12,0,1,11,2,True
3,H6_violation,12,12,0,9,3,2,True


## 11. Soporte mínimo para holdout agrupado

Condición lógica mínima para dejar al menos un grupo de cada clase fuera del entrenamiento sin eliminar esa clase del train:

```text
n_0 >= 2
n_1 >= 2
```

No se interpreta como suficiencia estadística; solo como posibilidad mínima de construir un holdout binario.


In [11]:
target_readiness = (
    master_v2_balance.copy()
)

target_readiness[
    "binary_seed_identifiable"
] = (
    target_readiness[
        "both_classes_observed"
    ]
)

target_readiness[
    "grouped_holdout_class_support"
] = (
    (
        target_readiness[
            "n_0"
        ]
        >=
        2
    )
    &
    (
        target_readiness[
            "n_1"
        ]
        >=
        2
    )
)

target_readiness[
    "baseline_status"
] = np.select(
    [
        target_readiness[
            "grouped_holdout_class_support"
        ],

        target_readiness[
            "binary_seed_identifiable"
        ]
    ],
    [
        "BASELINE_CANDIDATE",
        "BINARY_SEED_ONLY_HOLDOUT_BLOCKED"
    ],
    default="NOT_TRAINABLE"
)

display(
    target_readiness
)


,target,n_rows_catching,n_labeled,n_unknown,n_0,n_1,n_classes,both_classes_observed,binary_seed_identifiable,grouped_holdout_class_support,baseline_status
0,H1_violation,12,12,0,0,12,1,False,False,False,NOT_TRAINABLE
1,H4_violation,12,11,1,1,10,2,True,True,False,BINARY_SEED_ONLY_HOLDOUT_BLOCKED
2,H5_violation,12,12,0,1,11,2,True,True,False,BINARY_SEED_ONLY_HOLDOUT_BLOCKED
3,H6_violation,12,12,0,9,3,2,True,True,True,BASELINE_CANDIDATE


## 12. Reconstruir membresía L/U v2

In [12]:
membership_records = []

for _, row in (
    master_v2.iterrows()
):

    complete_x = bool(
        pd.notna(
            row.get(
                "structure_depth"
            )
        )
        and
        pd.notna(
            row.get(
                "interactive_control_count"
            )
        )
        and
        pd.notna(
            row.get(
                "interactive_label_ratio"
            )
        )
    )

    for target in TARGETS:

        y = row[
            target
        ]

        if not complete_x:

            role = (
                "EXCLUDED_INCOMPLETE_X"
            )

        elif pd.isna(
            y
        ):

            role = "U"

        elif y in [
            0,
            1
        ]:

            role = "L"

        else:

            role = (
                "EXCLUDED_INVALID_Y"
            )

        membership_records.append({
            "master_instance_id":
                row[
                    "master_instance_id"
                ],

            "source_dataset":
                row[
                    "source_dataset"
                ],

            "source_instance_id":
                row[
                    "source_instance_id"
                ],

            "group_id":
                row[
                    "group_id"
                ],

            "source_match_level":
                row[
                    "source_match_level"
                ],

            "target":
                target,

            "observed_y":
                y,

            "ssl_role":
                role
        })

membership_v2 = pd.DataFrame(
    membership_records
)

split_summary_v2 = (
    membership_v2
    .groupby(
        [
            "target",
            "ssl_role"
        ],
        dropna=False
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

display(
    split_summary_v2
)


ssl_role,target,EXCLUDED_INCOMPLETE_X,L,U
0,H1_violation,2,12,998
1,H4_violation,2,11,999
2,H5_violation,2,12,998
3,H6_violation,2,12,998


## 13. ¿Qué aportaban las filas excluidas?

Este análisis es clave para decidir si hace falta una recuperación dinámica.

Para cada fila excluida se muestra su vector Y real. Así puede verse si una fila no recuperada contiene precisamente la clase que falta para alguna heurística.


In [13]:
excluded_target_view_cols = [
    "site_feature",
    "negative_targets",
    "extract_reason",
    "static_max_dom_depth",
    "static_total_interactive_controls",
    "static_control_label_proxy_ratio"
] + TARGETS

excluded_target_view_cols = [
    col
    for col in excluded_target_view_cols
    if col in excluded_xy.columns
]

display(
    excluded_xy[
        excluded_target_view_cols
    ]
)


,site_feature,negative_targets,extract_reason,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,H1_violation,H4_violation,H5_violation,H6_violation
0,react-saas-template,H6_violation,NaN,2.0,0.0,NaN,1,1.0,1,0
1,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation | H4_violation | H5_violation | H6_violation,NO_HTML_FILES,NaN,NaN,NaN,0,0.0,0,0


## 14. Prioridad de recuperación por target

In [14]:
critical_records = []

for target in TARGETS:

    current = target_readiness[
        target_readiness[
            "target"
        ]
        ==
        target
    ].iloc[0]

    current_n0 = int(
        current[
            "n_0"
        ]
    )

    current_n1 = int(
        current[
            "n_1"
        ]
    )

    for _, row in (
        excluded_xy.iterrows()
    ):

        y = pd.to_numeric(
            pd.Series(
                [
                    row.get(
                        target,
                        np.nan
                    )
                ]
            ),
            errors="coerce"
        ).iloc[0]

        if pd.isna(
            y
        ):

            contribution = (
                "UNKNOWN_Y"
            )

        elif y == 0:

            contribution = (
                "ADDS_NEGATIVE"
            )

        elif y == 1:

            contribution = (
                "ADDS_POSITIVE"
            )

        else:

            contribution = (
                "INVALID_Y"
            )

        critical_for_holdout = bool(
            (
                current_n0
                <
                2
                and
                y
                ==
                0
            )
            or
            (
                current_n1
                <
                2
                and
                y
                ==
                1
            )
        )

        critical_records.append({
            "target":
                target,

            "excluded_site":
                row[
                    "site_feature"
                ],

            "observed_y":
                y,

            "contribution":
                contribution,

            "current_n_0":
                current_n0,

            "current_n_1":
                current_n1,

            "critical_for_minimum_grouped_holdout":
                critical_for_holdout,

            "extract_reason":
                row.get(
                    "extract_reason",
                    ""
                )
        })

critical_recovery = pd.DataFrame(
    critical_records
)

display(
    critical_recovery[
        critical_recovery[
            "critical_for_minimum_grouped_holdout"
        ]
    ]
)


,target,excluded_site,observed_y,contribution,current_n_0,current_n_1,critical_for_minimum_grouped_holdout,extract_reason
1,H1_violation,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,0.0,ADDS_NEGATIVE,0,12,True,NO_HTML_FILES
3,H4_violation,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,0.0,ADDS_NEGATIVE,1,10,True,NO_HTML_FILES
5,H5_violation,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,0.0,ADDS_NEGATIVE,1,11,True,NO_HTML_FILES


## 15. Decisión de continuidad

Posibles estados:

```text
PROCEED_TO_28_FULL_BASELINE
PROCEED_TO_28_PARTIAL_BASELINE
MASTER_V2_VALID_BUT_NO_TARGET_HAS_HOLDOUT_SUPPORT
```

Además se registra si alguna fila incompleta es crítica para desbloquear targets.


In [15]:
full_baseline_ready = bool(
    target_readiness[
        "grouped_holdout_class_support"
    ].all()
)

ready_targets = (
    target_readiness.loc[
        target_readiness[
            "grouped_holdout_class_support"
        ],
        "target"
    ]
    .tolist()
)

blocked_targets = (
    target_readiness.loc[
        ~target_readiness[
            "grouped_holdout_class_support"
        ],
        "target"
    ]
    .tolist()
)

critical_sites = (
    critical_recovery.loc[
        critical_recovery[
            "critical_for_minimum_grouped_holdout"
        ],
        "excluded_site"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if full_baseline_ready:

    global_status = (
        "PROCEED_TO_28_FULL_BASELINE"
    )

elif len(
    ready_targets
) > 0:

    global_status = (
        "PROCEED_TO_28_PARTIAL_BASELINE"
    )

else:

    global_status = (
        "MASTER_V2_VALID_BUT_NO_TARGET_HAS_HOLDOUT_SUPPORT"
    )

decision = pd.DataFrame([
    {
        "master_v2_built":
            master_v2_built,

        "master_v2_rows":
            len(
                master_v2
            ),

        "salvaged_negative_feature_rows":
            len(
                safe_extended
            ),

        "excluded_incomplete_negative_rows":
            len(
                excluded_negative_features
            ),

        "ready_targets":
            " | ".join(
                ready_targets
            ),

        "blocked_targets":
            " | ".join(
                blocked_targets
            ),

        "critical_excluded_sites":
            " | ".join(
                critical_sites
            ),

        "full_baseline_ready":
            full_baseline_ready,

        "global_status":
            global_status
    }
])

display(
    decision
)


,master_v2_built,master_v2_rows,salvaged_negative_feature_rows,excluded_incomplete_negative_rows,ready_targets,blocked_targets,critical_excluded_sites,full_baseline_ready,global_status
0,True,1012,10,2,H6_violation,H1_violation | H4_violation | H5_violation,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,False,PROCEED_TO_28_PARTIAL_BASELINE


## 16. Guardar artefactos

In [16]:
master_v2.to_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset_v2.csv",
    index=False
)

membership_v2.to_csv(
    SEMISUP_INTERIM_DIR
    / "semisupervised_target_membership_long_v2.csv",
    index=False
)

usable_xy.to_csv(
    SEMISUP_INTERIM_DIR
    / "catching_negative_xy_salvaged.csv",
    index=False
)

excluded_xy.to_csv(
    SEMISUP_INTERIM_DIR
    / "catching_negative_xy_excluded_incomplete.csv",
    index=False
)

master_v2_balance.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_label_balance.csv",
    index=False
)

target_readiness.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_target_readiness.csv",
    index=False
)

split_summary_v2.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_target_split_summary_v2.csv",
    index=False
)

critical_recovery.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_critical_recovery_by_target.csv",
    index=False
)

excluded_negative_features.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_excluded_negative_feature_rows.csv",
    index=False
)

collisions.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_collisions.csv",
    index=False
)

decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_salvage_decision.csv",
    index=False
)


In [17]:
expected_outputs = [
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset_v2.csv",

    SEMISUP_INTERIM_DIR
    / "semisupervised_target_membership_long_v2.csv",

    SEMISUP_INTERIM_DIR
    / "catching_negative_xy_salvaged.csv",

    SEMISUP_INTERIM_DIR
    / "catching_negative_xy_excluded_incomplete.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_label_balance.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_target_readiness.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_critical_recovery_by_target.csv",

    SEMISUP_REPORTS_DIR
    / "semisupervised_master_v2_salvage_decision.csv"
]

for path in expected_outputs:

    print(
        path.relative_to(
            PROJECT_DIR
        ),
        "✅"
        if path.exists()
        else "❌"
    )


data/semisupervised/interim/semisupervised_proxy_master_dataset_v2.csv ✅
data/semisupervised/interim/semisupervised_target_membership_long_v2.csv ✅
data/semisupervised/interim/catching_negative_xy_salvaged.csv ✅
data/semisupervised/interim/catching_negative_xy_excluded_incomplete.csv ✅
reports/tables/semisupervised/semisupervised_master_v2_label_balance.csv ✅
reports/tables/semisupervised/semisupervised_master_v2_target_readiness.csv ✅
reports/tables/semisupervised/semisupervised_critical_recovery_by_target.csv ✅
reports/tables/semisupervised/semisupervised_master_v2_salvage_decision.csv ✅


## Punto de detención

No entrenar aún.

El siguiente paso depende exclusivamente de:

```text
semisupervised_master_v2_salvage_decision.csv
```

Si el resultado es `PROCEED_TO_28_PARTIAL_BASELINE`, el notebook 28 solo podrá entrenar/evaluar los targets marcados como listos.

Si una fila incompleta aparece como crítica para H1/H4/H5, la siguiente fase correcta será una recuperación técnica dirigida de esa interfaz dinámica, no inventar valores ni convertir `NaN` en cero.

El master v2 generado aquí sí puede considerarse un **master semisupervisado experimental válido para el espacio proxy**, porque incorpora únicamente filas con X observada/reproducible y conserva las exclusiones por separado.
